# TRNO 10-Q3 2024 prebuilt-layout markdown output

In [ ]:
import os, time
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.ai.contentunderstanding import ContentUnderstandingClient

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(REPO_ROOT / ".env")

raw_endpoint = os.environ["FOUNDRY_ENDPOINT"]
endpoint = raw_endpoint.split("/api/projects/")[0].rstrip("/") + "/"
credential = ClientSecretCredential(
    tenant_id=os.environ["AZURE_TENANT_ID"],
    client_id=os.environ["AZURE_CLIENT_ID"],
    client_secret=os.environ["AZURE_CLIENT_SECRET"],
)
client = ContentUnderstandingClient(endpoint=endpoint, credential=credential, api_version="2025-11-01")
print("client ready:", endpoint)

In [ ]:
pdf_path = REPO_ROOT / "email" / "attachements" / "trno_10Q_Q3_2024.pdf"
assert pdf_path.exists(), pdf_path

t0 = time.time()
poller = client.begin_analyze_binary(
    analyzer_id="prebuilt-layout",
    binary_input=pdf_path.read_bytes(),
    content_type="application/pdf",
)
result = poller.result().as_dict()
print(f"done in {time.time()-t0:.1f}s")

contents = result["contents"][0]
md = contents["markdown"]
print("markdown length:", len(md))
print("pages:", contents.get("startPageNumber"), "to", contents.get("endPageNumber"))
print("tables:", len(contents.get("tables", [])))

In [ ]:
out_dir = REPO_ROOT / "output"
out_dir.mkdir(exist_ok=True)
md_path = out_dir / "trno_10Q_Q3_2024.md"
md_path.write_text(md, encoding="utf-8")
print("wrote", md_path, f"({md_path.stat().st_size:,} bytes)")

In [ ]:
from IPython.display import Markdown, display
display(Markdown(md[:20000]))

## LLM Demo: indentation doesn't matter

Pass the **flat** (no-indentation) HTML table from `prebuilt-layout` to GPT and ask it to:
1. **Classify** each row as section header / line item / subtotal and assign indent levels
2. **Extract** structured numeric data for a specific period

If the LLM recovers the hierarchy correctly from the flat table, it proves indentation is unnecessary.

In [ ]:
import re, json
from openai import AzureOpenAI

aoai = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=credential.get_token("https://cognitiveservices.azure.com/.default").token,
    api_version="2024-12-01-preview",
)
MODEL = os.environ.get("GPT_MODEL_DEPLOYMENT", "gpt-4.1")

# Extract the Statements of Operations table from the markdown
# It's an HTML <table> whose caption contains "Statements of Operations"
tables_html = re.findall(r"<table>.*?</table>", md, re.DOTALL)
ops_table = [t for t in tables_html if "REVENUES" in t and "Rental revenues" in t][0]
print(f"Ops table: {len(ops_table):,} chars")
print(ops_table[:300], "...")

In [ ]:
# --- DEMO 1: Ask the LLM to recover the row hierarchy from the flat table ---

resp1 = aoai.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a financial document analyst."},
        {"role": "user", "content": f"""Below is an HTML table extracted via OCR from an SEC 10-Q filing.
The table has NO indentation - all rows are flat.

Reconstruct the hierarchy implied by the wording and ordering of the rows.
For each row, capture the row label, its nesting level within the statement,
and the kind of row it appears to be.

Any reasonable JSON structure is acceptable. Return JSON only.

TABLE:
{ops_table}"""}
    ],
    temperature=0,
)

parsed_hierarchy = json.loads(resp1.choices[0].message.content)
if isinstance(parsed_hierarchy, list):
    hierarchy = parsed_hierarchy
elif isinstance(parsed_hierarchy, dict):
    hierarchy = next(
        value for value in parsed_hierarchy.values()
        if isinstance(value, list) and value and isinstance(value[0], dict)
    )
else:
    raise TypeError(f"Unexpected hierarchy payload type: {type(parsed_hierarchy)}")

print(f"LLM classified {len(hierarchy)} rows:\n")

# Discover which keys the LLM chose
sample = next(row for row in hierarchy if isinstance(row, dict))

def find_json_key(obj, *hints, exclude=()):
    excluded = set(exclude)
    for key in obj:
        if key in excluded:
            continue
        key_lower = key.lower()
        if any(hint in key_lower for hint in hints):
            return key

    remaining = [key for key in obj if key not in excluded]
    if len(remaining) == 1:
        return remaining[0]

    raise KeyError(f"No key matching {hints} in {list(obj.keys())}")

k_label = find_json_key(sample, "label", "name", "text", "row")
k_level = find_json_key(sample, "level", "indent", "depth")
k_type = find_json_key(sample, "type", "role", "class", "category", "kind", "classification", exclude=(k_label, k_level))
print(f"LLM-chosen keys: label={k_label!r}, level={k_level!r}, type={k_type!r}\n")

for row in hierarchy:
    indent = "  " * int(row[k_level])
    label = str(row[k_label])[:55]
    print(f"{indent}{label:<55}  level={row[k_level]}  type={row[k_type]}")

In [ ]:
# --- DEMO 2: Ask the LLM to extract structured data from the flat table ---

resp2 = aoai.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a financial data extraction assistant. Return only valid JSON."},
        {"role": "user", "content": f"""From this SEC 10-Q Consolidated Statements of Operations table, pull out the headline figures for the current quarter and the comparable prior-year quarter shown in the table: revenues, total costs and expenses, net income, and basic EPS. Also capture the individual cost line items together with the section each one belongs to.

Any reasonable JSON structure is acceptable. Return JSON only.

TABLE:
{ops_table}"""}
    ],
    temperature=0,
)

extracted = json.loads(resp2.choices[0].message.content)
print(json.dumps(extracted, indent=2))

## Demo 3: Statement extraction from one markdown chunk

In practice, the LLM would receive one extracted markdown chunk for the statement section,
not a prompt that says it is looking at two separate tables.

This demo passes a single chunk from the Consolidated Statements of Equity section and asks
the model to interpret it as one statement, even though the extracted chunk contains multiple
internal table segments.

In [ ]:
# Extract one markdown chunk for the full equity statement section
match = re.search(
    r"# Terreno Realty Corporation Consolidated Statements of Equity.*?The accompanying condensed notes are an integral part of these consolidated financial statements\.",
    md,
    re.DOTALL,
)
assert match, "Could not find equity statement chunk in markdown"

equity_chunk = match.group(0)
print(f"Equity chunk length: {len(equity_chunk):,} chars")
print(equity_chunk[:1200], "...", sep="\n")

In [ ]:
# --- DEMO 3: LLM interprets one extracted markdown chunk for the equity statement ---

resp3 = aoai.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a financial data extraction assistant. Return only valid JSON."},
        {"role": "user", "content": f"""Below is a markdown chunk extracted from an SEC 10-Q filing.
It contains the Consolidated Statements of Equity section for Terreno Realty Corporation.

Interpret the chunk as one statement and extract, for each reported period, the opening total equity,
closing total equity, quarterly net income entries, total dividends, total proceeds from stock issuances,
and ending share count.

Preserve accounting sign conventions exactly as shown in the chunk. Amounts shown in parentheses are negative values, not positive ones. For example, dividends shown as (43,517) should be returned as -43517.

Any reasonable JSON structure is acceptable. Return JSON only.

CHUNK:
{equity_chunk}"""}
    ],
    temperature=0,
)

equity_extracted = json.loads(resp3.choices[0].message.content)
print(json.dumps(equity_extracted, indent=2))

In [ ]:
# --- Validate the extracted equity values against the statement ---

periods = equity_extracted["periods"]
p2024 = next(period for period in periods if "2024" in period["period_label"])
p2023 = next(period for period in periods if "2023" in period["period_label"])

net_income_2024 = [entry["net_income"] for entry in p2024["quarters"]]
net_income_2023 = [entry["net_income"] for entry in p2023["quarters"]]
dividends_2024 = sum(entry["total_dividends"] for entry in p2024["quarters"])
dividends_2023 = sum(entry["total_dividends"] for entry in p2023["quarters"])
issuance_2024 = sum(entry["total_proceeds_from_stock_issuances"] for entry in p2024["quarters"])
issuance_2023 = sum(entry["total_proceeds_from_stock_issuances"] for entry in p2023["quarters"])
ending_shares_2024 = p2024["quarters"][-1]["ending_share_count"]
ending_shares_2023 = p2023["quarters"][-1]["ending_share_count"]

checks = [
    ("Opening total equity 2024", p2024["opening_total_equity"], 2914627),
    ("Closing total equity 2024", p2024["closing_total_equity"], 3631295),
    ("Quarterly net income 2024", net_income_2024, [36059, 35696, 36639]),
    ("Total dividends 2024", dividends_2024, -135917),
    ("Stock issuance proceeds 2024", issuance_2024, 736414),
    ("Ending share count 2024", ending_shares_2024, 99227029),
    ("Opening total equity 2023", p2023["opening_total_equity"], 2229851),
    ("Closing total equity 2023", p2023["closing_total_equity"], 2736970),
    ("Quarterly net income 2023", net_income_2023, [23331, 40254, 30315]),
    ("Total dividends 2023", dividends_2023, -105099),
    ("Stock issuance proceeds 2023", issuance_2023, 509700),
    ("Ending share count 2023", ending_shares_2023, 84871366),
]

passed = 0
for name, got, expected in checks:
    ok = got == expected
    passed += int(ok)
    print(f"{name}: {'OK' if ok else f'NOT OK (got {got}, expected {expected})'}")

print(f"\n{passed}/{len(checks)} checks passed")